# RAG Application with OpenAI API for a Car Manual

This project builds a Retrieval-Augmented Generation (RAG) application that answers operational questions from a car manual PDF. The application uses LangChain, OpenAI embeddings, FAISS vector search, and a Gradio interface.



In [ ]:
!pip install -q langchain langchain-text-splitters langchain-community langchain-openai faiss-cpu pypdf gradio

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 46.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 98.6/98.6 kB 7.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 60.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 336.3/336.3 kB 14.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 45.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 542.4/542.4 kB 32.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.9/64.9 kB 4.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 3.3 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.33.1 which is incompatible.


## Step 1: Upload the Car Manual PDF

The official car manual PDF is uploaded and used as the source document for the RAG application.

In [28]:
from google.colab import files

uploaded = files.upload()

pdf_path = list(uploaded.keys())[0]
print("Uploaded file:", pdf_path)

Saving 2024 defender owner manual.pdf to 2024 defender owner manual (2).pdf
Uploaded file: 2024 defender owner manual (2).pdf


## Step 2: Load and Chunk the PDF

The PDF is loaded using PyPDFLoader and split into smaller text chunks. The required chunk size is 800 with a chunk overlap of 100.

In [29]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

loader = PyPDFLoader(pdf_path)
documents = loader.load()

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=800,
    chunk_overlap=100
)

chunks = text_splitter.split_documents(documents)

print("Number of pages loaded:", len(documents))
print("Number of chunks created:", len(chunks))

Number of pages loaded: 244
Number of chunks created: 683


## Step 3: Set OpenAI API Key

The OpenAI API key is used to access the embedding model and the GPT model.

In [30]:

import os
from getpass import getpass

os.environ["OPENAI_API_KEY"] = getpass("Enter your OpenAI API key: ")

Enter your OpenAI API key: ··········


## Step 4: Create Embeddings and FAISS Vector Database

OpenAI embeddings are used to convert text chunks into vectors. FAISS stores the vectors and allows the system to retrieve relevant manual sections.

In [10]:
from langchain_openai import OpenAIEmbeddings
from langchain_community.vectorstores import FAISS

embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

vector_db = FAISS.from_documents(chunks, embeddings)

print("FAISS vector database created successfully.")

FAISS vector database created successfully.


## Step 5: Create the Car Manual QA Chain

This function retrieves relevant manual sections and answers the question using only the retrieved context. If the answer is not found in the manual, the system returns "Not found in manual."

In [12]:
vector_db = FAISS.from_documents(chunks, embeddings)


In [13]:
retriever = vector_db.as_retriever(search_kwargs={"k": 4})

In [17]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate

llm = ChatOpenAI(
    model="gpt-4o-mini",
    temperature=0
)

retriever = vector_db.as_retriever(search_kwargs={"k": 4})

prompt_template = """
Answer the question using ONLY the context below.
If the answer is not in the document, say "Not found in manual".
Use this tool to answer questions about vehicle operation and characteristics.
Return page numbers as citations.

Context:
{context}

Question:
{question}

Answer:
"""

prompt = ChatPromptTemplate.from_template(prompt_template)

def answer_question(question):
    retrieved_docs = retriever.invoke(question)

    context_text = ""
    page_numbers = []

    for doc in retrieved_docs:
        page = doc.metadata.get("page", "Unknown")
        page_numbers.append(page + 1 if isinstance(page, int) else page)
        context_text += f"\nPage {page + 1 if isinstance(page, int) else page}:\n{doc.page_content}\n"

    chain = prompt | llm
    response = chain.invoke({
        "context": context_text,
        "question": question
    })

    return response.content

## Step 6: Test the RAG Model

The model is tested with operational questions from the car manual.

In [24]:
import os

print(os.listdir())

['.config', '2024 defender owner manual (1).pdf', '2024 defender owner manual.pdf', 'sample_data']


In [25]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

loader = PyPDFLoader(pdf_path)
documents = loader.load()

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=800,
    chunk_overlap=100
)

chunks = text_splitter.split_documents(documents)

print("Number of pages loaded:", len(documents))
print("Number of chunks created:", len(chunks))

Number of pages loaded: 99
Number of chunks created: 156


In [26]:
from langchain_openai import OpenAIEmbeddings
from langchain_community.vectorstores import FAISS

embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

vector_db = FAISS.from_documents(chunks, embeddings)

print("FAISS vector database created successfully.")

FAISS vector database created successfully.


In [43]:
import gradio as gr

def car_manual_chat(question):
    if question.strip() == "":
        return "Please enter a question."
    return answer_question(question)

app = gr.Interface(
    fn=car_manual_chat,
    inputs=gr.Textbox(label="Ask a question about the vehicle manual"),
    outputs=gr.Textbox(label="Answer"),
    title="Car Manual RAG Assistant",
    description="Ask operational questions about the uploaded vehicle manual. The assistant answers using only the manual."
)

app.launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://b290e86bb5e3a50a18.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
